# CLAIRE — Batch Pipelines

Self-contained PyTorch model (VAE + counterfactual-augmented classifier), no external
repo dependency. Each pipeline has a **TRIAL** cell (1-2 files, writes to a separate
`_TRIAL.csv`) before the full batch run — delete the TRIAL cells once confirmed working.

**Paths:**
- Synthetic data: `Data_generation/SF_and_Hidden_Nodes_Data_Simulation/results/generated_160_csv_10_seeds/full_data`
- Semi-synthetic data: `Data_generation/HR_Simulation_Datasets`
- Results output: `model_results/CLAIRE_syn_results.csv` and `model_results/CLAIRE_semi_syn_results.csv`


In [ ]:
# ==========================================
# LIBRARIES
# ==========================================
import os
import glob
import re
import warnings

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings('ignore')

# Shared paths (relative to this notebook's location: Fairness_models/)
SYN_INPUT_FOLDER = "../Data_generation/SF_and_Hidden_Nodes_Data_Simulation/results/generated_160_csv_10_seeds/full_data"
SEMI_INPUT_FOLDER = "../Data_generation/HR_Simulation_Datasets"
OUTPUT_DIR = "../model_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## CLAIRE Architectures

In [ ]:
# ==========================================
# STEP 1: CLAIRE ARCHITECTURES
# ==========================================
def compute_mmd(z, s):
    """ Maximum Mean Discrepancy (MMD) Loss to remove bias from Latent Space Z """
    s_squeeze = s.squeeze()
    z_0 = z[s_squeeze == 0]
    z_1 = z[s_squeeze == 1]

    if len(z_0) == 0 or len(z_1) == 0:
        return torch.tensor(0.0, device=z.device)

    def gaussian_kernel(a, b, sigma=1.0):
        a_size = a.size(0)
        b_size = b.size(0)
        a = a.unsqueeze(1).expand(a_size, b_size, -1)
        b = b.unsqueeze(0).expand(a_size, b_size, -1)
        dist = torch.pow(a - b, 2).sum(2)
        return torch.exp(-dist / (2 * sigma ** 2))

    k_00 = gaussian_kernel(z_0, z_0).mean()
    k_11 = gaussian_kernel(z_1, z_1).mean()
    k_01 = gaussian_kernel(z_0, z_1).mean()

    return k_00 + k_11 - 2 * k_01

class CLAIRE_VAE(nn.Module):
    """ Phase 1: Representation Learning """
    def __init__(self, x_dim, z_dim=5, hidden_dim=64):
        super(CLAIRE_VAE, self).__init__()
        # Encoder takes X and S
        self.enc_fc1 = nn.Linear(x_dim + 1, hidden_dim)
        self.enc_fc2_mu = nn.Linear(hidden_dim, z_dim)
        self.enc_fc2_logvar = nn.Linear(hidden_dim, z_dim)

        # Decoder takes Z and S to reconstruct X
        self.dec_fc1 = nn.Linear(z_dim + 1, hidden_dim)
        self.dec_fc2_x = nn.Linear(hidden_dim, x_dim)

    def encode(self, x, s):
        h = F.relu(self.enc_fc1(torch.cat([x, s], dim=1)))
        return self.enc_fc2_mu(h), self.enc_fc2_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, s):
        h = F.relu(self.dec_fc1(torch.cat([z, s], dim=1)))
        return self.dec_fc2_x(h)

    def forward(self, x, s):
        mu, logvar = self.encode(x, s)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z, s)
        return recon_x, z, mu, logvar

class CLAIRE_Classifier(nn.Module):
    """ Phase 2: Main Predictive Model """
    def __init__(self, x_dim, hidden_dim=64):
        super(CLAIRE_Classifier, self).__init__()
        self.fc1 = nn.Linear(x_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        return torch.sigmoid(self.out(h))


# ==========================================
# STEP 2: REPRESENTATION LEARNING LOOP
# ==========================================
def train_claire_vae(vae_model, X_tensor, S_tensor, epochs=50, batch_size=128, lr=1e-3, mmd_weight=10.0):
    vae_model.train()
    optimizer = optim.Adam(vae_model.parameters(), lr=lr)

    dataset = TensorDataset(X_tensor, S_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        for batch_x, batch_s in dataloader:
            optimizer.zero_grad()

            recon_x, z, mu, logvar = vae_model(batch_x, batch_s)

            recon_loss = F.mse_loss(recon_x, batch_x, reduction='sum')
            kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
            mmd_loss = compute_mmd(z, batch_s)

            loss = recon_loss + kld_loss + (mmd_weight * mmd_loss)
            loss.backward()
            optimizer.step()

    return vae_model


# ==========================================
# STEP 3: COUNTERFACTUAL AUGMENTATION
# ==========================================
def train_claire_classifier(vae_model, classifier, X_tensor, S_tensor, Y_tensor,
                            epochs=100, batch_size=128, lr=1e-3, penalty_weight=1.0):
    vae_model.eval()
    classifier.train()
    optimizer = optim.Adam(classifier.parameters(), lr=lr)

    dataset = TensorDataset(X_tensor, S_tensor, Y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        for batch_x, batch_s, batch_y in dataloader:
            optimizer.zero_grad()

            # --- GENERATE COUNTERFACTUAL TWINS ---
            with torch.no_grad():
                _, z, _, _ = vae_model(batch_x, batch_s)
                batch_s_cf = 1.0 - batch_s  # Flip S
                x_cf = vae_model.decode(z, batch_s_cf)

            # --- TRAIN THE CLASSIFIER ---
            y_pred_real = classifier(batch_x)
            y_pred_cf = classifier(x_cf)

            loss_cls = F.binary_cross_entropy(y_pred_real, batch_y)
            loss_cf = F.mse_loss(y_pred_real, y_pred_cf)  # Consistency Penalty

            loss = loss_cls + (penalty_weight * loss_cf)
            loss.backward()
            optimizer.step()

    return classifier

def predict_claire(classifier, X_tensor):
    classifier.eval()
    with torch.no_grad():
        probs = classifier(X_tensor)
        preds = (probs >= 0.5).float()
        return probs.numpy().flatten(), preds.numpy().flatten()


## Purely Synthetic Data

### 🧪 TRIAL — run on 1-2 files only (delete this cell once confirmed working)

In [ ]:
# --- TRIAL: synthetic, 2 files only, writes to a separate _TRIAL.csv ---
N_TRIAL_FILES = 2
trial_output_file = os.path.join(OUTPUT_DIR, "CLAIRE_syn_results_TRIAL.csv")

dataset_files_trial = glob.glob(os.path.join(SYN_INPUT_FOLDER, "*.csv"))[:N_TRIAL_FILES]
print(f"[TRIAL] Found {len(dataset_files_trial)} file(s) to test with.")

trial_results = []

for file_path in dataset_files_trial:
    dataset_name = os.path.basename(file_path)
    print(f"\n[TRIAL] Processing: {dataset_name} with CLAIRE...")

    try:
        df = pd.read_csv(file_path)

        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith(('U', 'H', 'C'))]

        if 'Y' not in df.columns or len(s_cols) == 0:
            print("  [SKIPPED] Missing 'Y' or 'S' columns.")
            continue

        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        S_data = df[[s_target]].values
        X_features = df[x_cols].values
        y_data = df['Y'].values.reshape(-1, 1)

        train_size = min(1000, int(len(X_features) * 0.8))

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        S_train_t = torch.tensor(S_train, dtype=torch.float32)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)

        x_dim = X_train.shape[1]
        z_dim = 5

        vae_model = CLAIRE_VAE(x_dim=x_dim, z_dim=z_dim)
        vae_model = train_claire_vae(vae_model, X_train_t, S_train_t, epochs=50, mmd_weight=10.0)

        classifier = CLAIRE_Classifier(x_dim=x_dim)
        classifier = train_claire_classifier(
            vae_model, classifier, X_train_t, S_train_t, y_train_t,
            epochs=100, penalty_weight=1.0
        )

        prob_preds, predictions = predict_claire(classifier, X_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        trial_results.append({
            "model_name": "CLAIRE", "name_dataset": dataset_name,
            "n_S": len(s_cols), "n_X": len(x_cols), "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4), "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4), "Pos_Rate_S0": round(rate_0, 4)
        })
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

trial_df = pd.DataFrame(trial_results)
trial_df.to_csv(trial_output_file, index=False)
print(f"\n[TRIAL] Done. Results written to {trial_output_file}")
trial_df


### Full batch pipeline — Purely Synthetic Data

In [ ]:
# ==========================================
# STEP 4: SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = SYN_INPUT_FOLDER
output_file = os.path.join(OUTPUT_DIR, "CLAIRE_syn_results.csv")

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with CLAIRE...")

    try:
        df = pd.read_csv(file_path)

        # --- FEATURE DISCOVERY ---
        s_cols = [col for col in df.columns if col.startswith('S')]
        x_cols = [col for col in df.columns if col.startswith('X')]
        u_cols = [col for col in df.columns if col.startswith(('U', 'H', 'C'))]

        if 'Y' not in df.columns or len(s_cols) == 0:
            print(f"  [SKIPPED] Missing 'Y' or 'S' columns.")
            continue

        s_target = 'S0' if 'S0' in s_cols else s_cols[0]
        S_data = df[[s_target]].values
        X_features = df[x_cols].values
        y_data = df['Y'].values.reshape(-1, 1)

        # Limit to 1000 samples to match FairPFN limits for fair comparison
        train_size = min(1000, int(len(X_features) * 0.8))

        # Split Data (S is included explicitly for training in CLAIRE)
        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        # Convert to Tensors
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        S_train_t = torch.tensor(S_train, dtype=torch.float32)
        X_test_t  = torch.tensor(X_test, dtype=torch.float32)

        x_dim = X_train.shape[1]
        z_dim = 5

        # --- PHASE 1: VAE REPRESENTATION LEARNING ---
        vae_model = CLAIRE_VAE(x_dim=x_dim, z_dim=z_dim)
        vae_model = train_claire_vae(vae_model, X_train_t, S_train_t, epochs=50, mmd_weight=10.0)

        # --- PHASE 2: CONSISTENCY CLASSIFIER ---
        classifier = CLAIRE_Classifier(x_dim=x_dim)
        classifier = train_claire_classifier(
            vae_model, classifier, X_train_t, S_train_t, y_train_t,
            epochs=100, penalty_weight=1.0
        )

        # --- PHASE 3: EVALUATION ---
        prob_preds, predictions = predict_claire(classifier, X_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        # Prediction Metrics
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        # Fairness Metrics
        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)

        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0

        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        # --- RECORD DATA ---
        master_results.append({
            "model_name": "CLAIRE",
            "name_dataset": dataset_name,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })

        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

# ==========================================
# 5. SAVE AGGREGATED RESULTS
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")


## Semi-Synthetic (HR) Data

### 🧪 TRIAL — run on 1-2 files only (delete this cell once confirmed working)

In [ ]:
# --- TRIAL: semi-synthetic, 2 files only, writes to a separate _TRIAL.csv ---
N_TRIAL_FILES = 2
trial_output_file_semi = os.path.join(OUTPUT_DIR, "CLAIRE_semi_syn_results_TRIAL.csv")

dataset_files_trial_semi = glob.glob(os.path.join(SEMI_INPUT_FOLDER, "*.csv"))[:N_TRIAL_FILES]
print(f"[TRIAL] Found {len(dataset_files_trial_semi)} file(s) to test with.")

trial_results_semi = []

for file_path in dataset_files_trial_semi:
    dataset_name = os.path.basename(file_path)
    print(f"\n[TRIAL] Processing: {dataset_name} with CLAIRE...")

    try:
        df = pd.read_csv(file_path)

        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values.reshape(-1, 1)

        s_cols = [col for col in df.columns if col.startswith('S') or col.startswith('G')]
        if not s_cols:
            print("  [SKIPPED] No 'S' or 'G' column found. CLAIRE requires a protected attribute for counterfactuals.")
            continue

        s_target = s_cols[0]
        S_data = df[[s_target]].values

        x_cols = [col for col in df.columns if col.startswith('X_')]
        X_features = df[x_cols].values

        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]

        train_size = min(1000, int(len(X_features) * 0.8))

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        S_train_t = torch.tensor(S_train, dtype=torch.float32)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)

        x_dim = X_train.shape[1]
        z_dim = 5

        vae_model = CLAIRE_VAE(x_dim=x_dim, z_dim=z_dim)
        vae_model = train_claire_vae(vae_model, X_train_t, S_train_t, epochs=50, mmd_weight=10.0)

        classifier = CLAIRE_Classifier(x_dim=x_dim)
        classifier = train_claire_classifier(
            vae_model, classifier, X_train_t, S_train_t, y_train_t,
            epochs=100, penalty_weight=1.0
        )

        prob_preds, predictions = predict_claire(classifier, X_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)
        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0
        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        trial_results_semi.append({
            "model_name": "CLAIRE", "name_dataset": dataset_name,
            "bias_level": bias_level, "threshold": threshold, "data_type": data_type,
            "n_S": len(s_cols), "n_X": len(x_cols), "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4), "Accuracy": round(acc, 4),
            "Precision": round(prec, 4), "Recall": round(rec, 4), "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4), "Pos_Rate_S0": round(rate_0, 4)
        })
        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

trial_df_semi = pd.DataFrame(trial_results_semi)
trial_df_semi.to_csv(trial_output_file_semi, index=False)
print(f"\n[TRIAL] Done. Results written to {trial_output_file_semi}")
trial_df_semi


### Full batch pipeline — Semi-Synthetic (HR) Data

In [ ]:
# ==========================================
# STEP 4: SEMI-SYNTHETIC PIPELINE INTEGRATION
# ==========================================
input_folder = SEMI_INPUT_FOLDER
output_file = os.path.join(OUTPUT_DIR, "CLAIRE_semi_syn_results.csv")

dataset_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(dataset_files)} semi-synthetic datasets to process.")

master_results = []

for file_path in dataset_files:
    dataset_name = os.path.basename(file_path)
    print(f"\nProcessing: {dataset_name} with CLAIRE...")

    try:
        df = pd.read_csv(file_path)

        # --- FILENAME PARAMETER EXTRACTION ---
        name_no_ext = dataset_name.replace('.csv', '')
        bias_level = "HighBias" if "HighBias" in name_no_ext else ("LowBias" if "LowBias" in name_no_ext else "Unknown")
        thresh_match = re.search(r'Thresh([\d\.]+)', name_no_ext)
        threshold = thresh_match.group(1) if thresh_match else "Unknown"
        data_type = "BIASED_ALL" if "BIASED_ALL" in name_no_ext else ("FAIR_ALL" if "FAIR_ALL" in name_no_ext else "Unknown")

        # --- DYNAMIC FEATURE DISCOVERY ---
        y_cols = [col for col in df.columns if col.startswith('Y')]
        if not y_cols:
            print("  [SKIPPED] Missing 'Y' target column.")
            continue
        y_data = df[y_cols[0]].values.reshape(-1, 1)

        # Protected Attributes (S and G)
        s_cols = [col for col in df.columns if col.startswith('S') or col.startswith('G')]
        if not s_cols:
            # CLAIRE requires an S column to perform its counterfactual flips
            print("  [SKIPPED] No 'S' or 'G' column found. CLAIRE requires a protected attribute for counterfactuals.")
            continue

        s_target = s_cols[0]
        S_data = df[[s_target]].values

        x_cols = [col for col in df.columns if col.startswith('X_')]
        X_features = df[x_cols].values

        u_cols = [col for col in df.columns if col.startswith(('U_', 'H_', 'C_'))]

        train_size = min(1000, int(len(X_features) * 0.8))

        # Split Data
        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_features, y_data, S_data, train_size=train_size, random_state=42, stratify=y_data
        )

        # Convert to Tensors
        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        S_train_t = torch.tensor(S_train, dtype=torch.float32)
        X_test_t  = torch.tensor(X_test, dtype=torch.float32)

        x_dim = X_train.shape[1]
        z_dim = 5

        # --- PHASE 1: VAE REPRESENTATION LEARNING ---
        vae_model = CLAIRE_VAE(x_dim=x_dim, z_dim=z_dim)
        vae_model = train_claire_vae(vae_model, X_train_t, S_train_t, epochs=50, mmd_weight=10.0)

        # --- PHASE 2: CONSISTENCY CLASSIFIER ---
        classifier = CLAIRE_Classifier(x_dim=x_dim)
        classifier = train_claire_classifier(
            vae_model, classifier, X_train_t, S_train_t, y_train_t,
            epochs=100, penalty_weight=1.0
        )

        # --- PHASE 3: EVALUATION ---
        prob_preds, predictions = predict_claire(classifier, X_test_t)
        y_test_flat = y_test.flatten()
        S_test_flat = S_test.flatten()

        # Prediction Metrics
        auc = roc_auc_score(y_test_flat, prob_preds)
        acc = accuracy_score(y_test_flat, predictions)
        prec = precision_score(y_test_flat, predictions, zero_division=0)
        rec = recall_score(y_test_flat, predictions, zero_division=0)
        f1 = f1_score(y_test_flat, predictions, zero_division=0)

        # Fairness Metrics
        group_1_mask = (S_test_flat == 1)
        group_0_mask = (S_test_flat == 0)

        rate_1 = np.mean(predictions[group_1_mask]) if np.sum(group_1_mask) > 0 else 0
        rate_0 = np.mean(predictions[group_0_mask]) if np.sum(group_0_mask) > 0 else 0
        stat_parity_diff = abs(rate_1 - rate_0)
        disp_impact = (rate_1 / rate_0) if rate_0 > 0 else float('inf')

        y_true_1, y_pred_1 = y_test_flat[group_1_mask], predictions[group_1_mask]
        tpr_1 = recall_score(y_true_1, y_pred_1, zero_division=0) if len(y_true_1) > 0 else 0

        y_true_0, y_pred_0 = y_test_flat[group_0_mask], predictions[group_0_mask]
        tpr_0 = recall_score(y_true_0, y_pred_0, zero_division=0) if len(y_true_0) > 0 else 0
        equal_opp_diff = abs(tpr_1 - tpr_0)

        # --- RECORD DATA ---
        master_results.append({
            "model_name": "CLAIRE",
            "name_dataset": dataset_name,
            "bias_level": bias_level,
            "threshold": threshold,
            "data_type": data_type,
            "n_S": len(s_cols),
            "n_X": len(x_cols),
            "n_U": len(u_cols),
            "total_samples": len(df),
            "ROC_AUC": round(auc, 4),
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1_Score": round(f1, 4),
            "Statistical_Parity_Diff_(ATE)": round(stat_parity_diff, 4),
            "Disparate_Impact_Ratio": round(disp_impact, 4),
            "Equal_Opportunity_Diff": round(equal_opp_diff, 4),
            "Pos_Rate_S1": round(rate_1, 4),
            "Pos_Rate_S0": round(rate_0, 4)
        })

        print(f"  [SUCCESS] AUC: {auc:.3f} | ATE: {stat_parity_diff:.3f}")

    except Exception as e:
        print(f"  [ERROR] Failed to process {dataset_name}. Reason: {str(e)}")

# ==========================================
# 5. SAVE AGGREGATED RESULTS
# ==========================================
if len(master_results) > 0:
    results_df = pd.DataFrame(master_results)
    file_exists = os.path.isfile(output_file)
    results_df.to_csv(output_file, mode='a', header=not file_exists, index=False)
    print("\n" + "="*50)
    print(f"ALL JOBS COMPLETE! Processed {len(master_results)} datasets.")
    print(f"Results appended to: {output_file}")
    print("="*50)
else:
    print("\nNo datasets were successfully processed.")
